# درس ۱۷ - ساخت عامل‌های هوش مصنوعی محلی با Foundry Local و Qwen

در این نوت‌بوک شما یک **دستیار مهندسی محلی** می‌سازید که به‌طور کامل روی ایستگاه کاری شما اجرا می‌شود. در هیچ مرحله‌ای از استنتاج ابری استفاده نمی‌شود. این دستیار می‌تواند:

۱. **ابزارها را فراخوانی کند** از طریق فراخوانی تابع Qwen به وسیله Foundry Local.
۲. **فایل‌ها را لیست کند و بخواند** درون یک دایرکتوری پروژه محصور شده.
۳. **کد را تحلیل کند** با معیارهای ساده.
۴. **مستندات را جستجو کند** با RAG محلی (Chroma).
۵. **از یک سرور MCP محلی استفاده کند** (اگر هیچ سروری پیکربندی نشده باشد، به‌صورت نرم از آن عبور می‌شود).

کد عامل تقریباً مشابه درس‌های ابری است — تنها نقطه اتصال کلاینت از ابر به `localhost` تغییر می‌کند.


## راه‌اندازی

قبل از اجرای این نوت‌بوک:

1. **Microsoft Foundry Local را نصب کنید** (برای سیستم‌عامل خود به [مستندات](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) مراجعه کنید).
2. **یک مدل Qwen را دانلود و اجرا کنید:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. بسته‌های پایتون زیر را نصب کنید.

همه چیز به صورت محلی اجرا می‌شود. یک سیستم با حافظه تقریبی ۸ گیگابایت حداقل واقعی است.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## ۱. اتصال به Foundry Local

`FoundryLocalManager` مدل را در صورت نیاز دانلود می‌کند، سرویس محلی را راه‌اندازی می‌کند، و به ما یک **نقطه‌پایانی سازگار با OpenAI** می‌دهد. سپس SDK استاندارد OpenAI را به آن اشاره می‌کنیم. کلید API یک جایگزین محلی است — هیچ اعتبار ابری در کار نیست.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## ۲. ابزارهای محلی (عملیات فایل با محیط جداشده)

ما یک پروژه نمونه کوچک روی دیسک ایجاد می‌کنیم، سپس ابزارهایی را تعریف می‌کنیم که محدوده‌ی آن‌ها به ریشه‌ی آن پروژه محدود است. بررسی محیط جداشده حتی به صورت محلی اهمیت دارد: ابزاری که مسیرهای دلخواه را می‌خواند با دسترسی‌های کاربر شما اجرا می‌شود و می‌تواند به هر چیزی که شما دسترسی دارید دست بزند.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. بازیابی محلی اسناد با Chroma  

ما مجموعه‌ای کوچک از قطعات مستندات را در یک مجموعه محلی Chroma جاسازی می‌کنیم. Chroma به صورت درون‌فرآیندی اجرا می‌شود و وکتورها را روی دیسک ذخیره می‌کند — بدون سرور، بدون ابر. ابزار `search_docs` مرتبط‌ترین قطعات را برای یک پرس‌وجو بازیابی می‌کند.  


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## ۴. حلقه فراخوانی ابزار  

اکنون ابزارها را با استفاده از طرح ابزارهای OpenAI در مدل ثبت می‌کنیم و حلقه استاندارد فراخوانی ابزار را اجرا می‌کنیم — مدل درخواست یک ابزار می‌کند، ما آن را به‌صورت محلی اجرا می‌کنیم، نتیجه را بازمی‌گردانیم و این روند را تا زمانی که مدل پاسخ نهایی را ارائه دهد تکرار می‌کنیم. فراخوانی تابع قابل‌اطمینان Qwen چیزی است که این کار را روی دستگاه ممکن می‌کند.  


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## ۵. MCP محلی (اختیاری)

MCP یک سرویس ابری نیست، بلکه یک روش انتقال است — یک سرور MCP می‌تواند به عنوان یک فرآیند محلی از طریق `stdio` اجرا شود. سلول زیر نشان می‌دهد چگونه می‌توانید به یک سرور MCP محلی متصل شوید اگر یکی را پیکربندی کرده باشید (برای مثال یک سرور سیستم فایل). این سلول به طور نرم‌افزاری عبور می‌کند وقتی `LOCAL_MCP_COMMAND` تنظیم نشده باشد، بنابراین دفترچه یادداشت بدون آن به طور کامل اجرا می‌شود.

نکته امنیتی: سرور MCP محلی با مجوزهای کاربر شما اجرا می‌شود. آن را به یک دایرکتوری پروژه محدود کنید و خروجی‌های آن را قبل از اقدام بررسی کنید.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## خلاصه

شما یک دستیار مهندسی ساختید که کاملاً روی دستگاه شما اجرا می‌شود:

- **Foundry Local** یک مدل **Qwen** را پشت یک نقطه پایانی سازگار با OpenAI ارائه داد — بنابراین کد عامل با درس‌های ابری همخوانی دارد.
- **ابزارهای ایزوله** به عامل دسترسی به فایل و تحلیل کد دادند بدون اینکه پروژه را ترک کند.
- **Chroma** ارائه‌دهنده **RAG محلی** روی مستندات بود.
- **Local MCP** نشان داد چگونه اکوسیستم MCP را به صورت آفلاین استفاده مجدد کنیم.

در هیچ مرحله‌ای از استنتاج ابری استفاده نشد.

### چالش

عامل محلی را گسترش دهید تا:

1. **با چند سرور MCP کار کند** — یک سرور فایل‌سیستم و یک سرور گیت متصل کنید و اجازه دهید عامل بین آن‌ها انتخاب کند.
2. **از حافظه محلی استفاده کند** — تاریخچه کوتاه مکالمه را روی دیسک ذخیره کنید تا دستیار در بازراه‌اندازی‌های نوت‌بوک گفتگوهای قبلی را به یاد داشته باشد.
3. **پشتیبانی از ارکستراسیون چندعامله محلی** — یک عامل محلی دوم (مثلاً یک بازبین) اضافه کنید و اجازه دهید این دو روی یک وظیفه همکاری کنند.

در درس بعدی یاد می‌گیرید چگونه عامل‌های هوش مصنوعی مستقر شده را ایمن کنید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
